# Session linearization

This notebook turns the filled environmental time series into the final linear ML dataset. Sessions are used only during construction: rows are grouped by separate room/source keys, split when a configurable time gap is exceeded, and compressed through overlapping lookback windows. The saved final dataset intentionally drops source, location, session, record, and timestamp metadata so every row stands alone.

In [1]:
from pathlib import Path
import sys

import pandas as pd

MAL_DIR = Path.cwd()
if MAL_DIR.name != "MAL":
    MAL_DIR = next(path for path in [Path.cwd(), *Path.cwd().parents] if path.name == "MAL")

sys.path.insert(0, str(MAL_DIR))

from scripts.linearize_session_windows import build_linearized_windows

INPUT_PATH = MAL_DIR / "data" / "processed" / "unified_environment_respondent_focus_score_filled.csv"
OUTPUT_PATH = MAL_DIR / "data" / "processed" / "linearized_session_windows_30min.csv"
REPORT_PATH = MAL_DIR / "data" / "processed" / "linearized_session_windows_30min_report.csv"

FEATURE_COLUMNS = ["temperature", "humidity", "light", "noise", "co2"]
GROUP_COLUMNS = ["source", "location_id"]
TARGET_COLUMN = "focus_score"
WINDOW_MINUTES = 30
SESSION_GAP_MINUTES = 30
METADATA_COLUMNS = [
    "source", "location_id", "linear_session_id", "window_end", "window_start",
    "source_record_id", "source_session_id", "session_elapsed_minutes",
]

INPUT_PATH, OUTPUT_PATH, REPORT_PATH

(WindowsPath('c:/Users/piotr/Desktop/University/Semester4/SEP4/SEP4/MAL/data/processed/unified_environment_respondent_focus_score_filled.csv'),
 WindowsPath('c:/Users/piotr/Desktop/University/Semester4/SEP4/SEP4/MAL/data/processed/linearized_session_windows_30min.csv'),
 WindowsPath('c:/Users/piotr/Desktop/University/Semester4/SEP4/SEP4/MAL/data/processed/linearized_session_windows_30min_report.csv'))

## Generate or load the final linearized dataset

In [2]:
if OUTPUT_PATH.exists() and REPORT_PATH.exists():
    linearized_df = pd.read_csv(OUTPUT_PATH, low_memory=False)
    session_report = pd.read_csv(REPORT_PATH, low_memory=False)
else:
    source_df = pd.read_csv(INPUT_PATH, low_memory=False)
    linearized_df, session_report = build_linearized_windows(
        source_df,
        feature_columns=FEATURE_COLUMNS,
        target_column=TARGET_COLUMN,
        group_columns=GROUP_COLUMNS,
        window_minutes=WINDOW_MINUTES,
        session_gap_minutes=SESSION_GAP_MINUTES,
    )
    linearized_df.to_csv(OUTPUT_PATH, index=False)
    session_report.to_csv(REPORT_PATH, index=False)

run_summary = pd.DataFrame([
    {
        "input_csv": str(INPUT_PATH),
        "output_csv": str(OUTPUT_PATH),
        "rows": len(linearized_df),
        "columns": linearized_df.shape[1],
        "sessions_used_during_build": len(session_report),
        "window_minutes": WINDOW_MINUTES,
        "session_gap_minutes": SESSION_GAP_MINUTES,
        "target_missing": int(linearized_df["focus_score"].isna().sum()),
        "target_min": int(linearized_df["focus_score"].min()),
        "target_max": int(linearized_df["focus_score"].max()),
    }
])
run_summary

,input_csv,output_csv,rows,columns,sessions_used_during_build,window_minutes,session_gap_minutes,target_missing,target_min,target_max
0,c:\Users\piotr\Desktop\University\Semester4\SE...,c:\Users\piotr\Desktop\University\Semester4\SE...,90453,36,167,30,30,0,1,5


## Session checks kept in the separate report

In [3]:
session_report[["rows", "duration_minutes"]].describe()

,rows,duration_minutes
count,167.000000,167.000000
mean,5676.035928,16226.236327
std,5364.559688,30745.961164
min,1.000000,0.000000
25%,3684.000000,5460.000000
50%,5461.000000,5461.000000
75%,5462.000000,5461.000000
max,45843.000000,240030.916667


In [4]:
session_report.sort_values("duration_minutes", ascending=False).head(10)

,linear_session_id,source,location_id,rows,first_timestamp,last_timestamp,duration_minutes
23,homecoach_5min_2025__homecoach_5min_2025_unkno...,homecoach_5min_2025,homecoach_5min_2025_unknown_location,45843,2025-07-18T07:28:36Z,2025-12-31T23:59:31Z,240030.916667
15,homecoach_5min_2024__homecoach_5min_2024_unkno...,homecoach_5min_2024,homecoach_5min_2024_unknown_location,25171,2024-04-11T10:48:03Z,2024-07-12T10:03:34Z,132435.516667
10,homecoach_5min_2023__homecoach_5min_2023_unkno...,homecoach_5min_2023,homecoach_5min_2023_unknown_location,24683,2023-08-20T07:53:47Z,2023-11-17T09:44:16Z,128270.483333
20,homecoach_5min_2025__homecoach_5min_2025_unkno...,homecoach_5min_2025,homecoach_5min_2025_unknown_location,24369,2025-01-01T00:02:30Z,2025-03-30T01:57:59Z,126835.483333
21,homecoach_5min_2025__homecoach_5min_2025_unkno...,homecoach_5min_2025,homecoach_5min_2025_unknown_location,20823,2025-03-30T03:02:59Z,2025-06-14T03:13:24Z,109450.416667
19,homecoach_5min_2024__homecoach_5min_2024_unkno...,homecoach_5min_2024,homecoach_5min_2024_unknown_location,16113,2024-11-03T18:09:12Z,2024-12-31T23:54:31Z,83865.316667
162,room_measurements__room_2_12__gap30min__s00001,room_measurements,room_2_12,15989,2025-09-15T00:00:03Z,2025-11-09T22:56:33Z,80576.500000
160,room_measurements__lab_4_2__gap30min__s00001,room_measurements,lab_4_2,15670,2025-09-15T11:51:39Z,2025-11-09T22:59:11Z,79867.533333
164,room_measurements__room_2_5__gap30min__s00001,room_measurements,room_2_5,14517,2025-09-15T11:43:27Z,2025-11-05T16:58:15Z,73754.800000
12,homecoach_5min_2024__homecoach_5min_2024_unkno...,homecoach_5min_2024,homecoach_5min_2024_unknown_location,12646,2024-01-01T00:02:30Z,2024-02-15T17:52:45Z,65870.250000


## Final dataset checks

In [5]:
metadata_check = pd.DataFrame([
    {"column": column, "present_in_final_dataset": column in linearized_df.columns}
    for column in METADATA_COLUMNS
])
metadata_check

,column,present_in_final_dataset
0,source,False
1,location_id,False
2,linear_session_id,False
3,window_end,False
4,window_start,False
5,source_record_id,False
6,source_session_id,False
7,session_elapsed_minutes,False


In [6]:
target_distribution = (
    linearized_df["focus_score"]
    .value_counts()
    .sort_index()
    .rename_axis("focus_score")
    .reset_index(name="rows")
)
target_distribution

,focus_score,rows
0,1,1290
1,2,10777
2,3,43229
3,4,23368
4,5,11789


In [7]:
feature_coverage = pd.DataFrame([
    {
        "feature": feature,
        "latest_non_null": int(linearized_df[f"{feature}_latest"].notna().sum()),
        "window_count_nonzero": int((linearized_df[f"{feature}_count"] > 0).sum()),
        "mean_non_null": int(linearized_df[f"{feature}_mean"].notna().sum()),
    }
    for feature in FEATURE_COLUMNS
])
feature_coverage

,feature,latest_non_null,window_count_nonzero,mean_non_null
0,temperature,90453,90453,90453
1,humidity,90453,90453,90453
2,light,90453,90453,90453
3,noise,90453,90453,90453
4,co2,90453,90453,90453


In [8]:
assert linearized_df["focus_score"].notna().all()
assert linearized_df["focus_score"].between(1, 5).all()
assert not any(column in linearized_df.columns for column in METADATA_COLUMNS)
print("Final dataset is saved with only target + engineered feature columns.")

Final dataset is saved with only target + engineered feature columns.


## Preview of the holy grail dataset

In [9]:
linearized_df.head(20)

,focus_score,temperature_latest,temperature_mean,temperature_min,temperature_max,temperature_std,temperature_count,temperature_range,humidity_latest,humidity_mean,...,noise_std,noise_count,noise_range,co2_latest,co2_mean,co2_min,co2_max,co2_std,co2_count,co2_range
0,4,19.3,18.533333,17.7,19.3,6.623192e-01,6,1.6,3249.0,3211.333333,...,0.584052,6,1.426495,0.000443,0.000441,0.000434,0.000447,0.000005,6,0.000013
1,3,19.7,19.700000,19.5,19.8,1.264911e-01,6,0.3,2809.0,3026.333333,...,0.305899,6,0.963637,0.000723,0.000541,0.000448,0.000723,0.000110,6,0.000275
2,3,19.4,19.450000,19.4,19.6,8.366600e-02,6,0.2,2401.0,2551.166667,...,0.276514,6,0.699567,0.001179,0.001028,0.000835,0.001179,0.000134,6,0.000344
3,4,19.3,19.233333,19.0,19.4,1.632993e-01,6,0.4,2209.0,2256.833333,...,0.335663,6,0.826070,0.001261,0.001301,0.001238,0.001391,0.000057,6,0.000153
4,4,19.9,19.716667,19.5,19.9,1.471960e-01,6,0.4,2025.0,2100.833333,...,0.146968,6,0.447463,0.001239,0.001240,0.001188,0.001294,0.000042,6,0.000106
5,3,20.3,20.100000,19.9,20.3,1.414214e-01,6,0.4,2025.0,2025.000000,...,0.430837,6,1.268384,0.001096,0.001121,0.001071,0.001200,0.000053,6,0.000130
6,5,20.7,20.583333,20.4,20.7,1.169045e-01,6,0.3,2116.0,2116.000000,...,0.884970,6,2.426150,0.001032,0.001046,0.001005,0.001076,0.000026,6,0.000071
7,4,20.8,20.783333,20.7,20.9,7.527727e-02,6,0.2,2025.0,2085.666667,...,0.351918,6,0.945345,0.001015,0.001021,0.001002,0.001038,0.000013,6,0.000036
8,2,20.9,20.833333,20.8,20.9,5.163978e-02,6,0.1,1936.0,1965.666667,...,0.486201,6,1.280110,0.001272,0.001152,0.001000,0.001272,0.000108,6,0.000272
9,3,21.0,20.980000,20.9,21.0,4.472136e-02,5,0.1,1849.0,1901.200000,...,0.176954,5,0.500435,0.001433,0.001421,0.001359,0.001511,0.000057,5,0.000152
